# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution
This notebook provides a step-by-step tutorial for loading and exploring the FAIR^2 dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset is specified via a Croissant schema URL and contains tabular records for 77 cancer survivors with second primary colorectal cancer, including detailed clinicopathological and biomarker variables.

In [ ]:
# Ensure `mlcroissant` is installed
!pip install -q mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using [`mlcroissant`](https://github.com/mlcommons/croissant).

The Croissant schema URL describes the dataset structure and locations.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Croissant schema URL for the dataset
croissant_url = "https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json"

# Load dataset metadata and records
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset Name: {metadata.name}")
print(f"Description: {metadata.description}")
print(f"Published: {metadata.datePublished}")
print(f"Number of record sets detected: {len(metadata.record_sets)}")

## 2. Data Overview
Explore available record sets, their field definitions, and their unique `@id` identifiers. All references to record sets, fields, or columns will be by their Croissant `@id`.

**Tip:** The `record_sets` attribute in Croissant metadata lists available record sets. Each record set contains field information and their `@id`.

In [ ]:
# List all available record sets and their fields (referenced by @id)
if len(metadata.record_sets) == 0:
    print("No record sets available in the schema (metadata.record_sets is empty). {dataset.metadata.name}")
else:
    for rs in metadata.record_sets:
        print(f"\nRecord set: {rs['@id']}")
        print(f"  Name: {rs.get('name', 'N/A')}")
        print(f"  Description: {rs.get('description', 'N/A')}")
        print("  Fields:")
        fields = rs.get('fields', [])
        for field in fields:
            print(f"    {field['@id']}: {field.get('name', 'N/A')} (type: {field.get('dataType', 'N/A')})")

# Store record_set @ids for later use
record_set_ids = [rs['@id'] for rs in metadata.record_sets]
print("\nList of record set @id values:")
print(record_set_ids)

## 3. Data Extraction
Load data from a specific record set into a DataFrame for further analysis. We will use the record set and field `@id` from the previous overview.

If there are multiple record sets, we load them all by their `@id`.

In [ ]:
# Helper: load all record sets into pandas DataFrames, indexing by record set @id
dataframes = {}

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded {len(df)} records for record set '{record_set_id}'")

# For demonstration: show columns and preview for the first record set
if len(record_set_ids) > 0:
    example_rs = record_set_ids[0]
    print(f\nColumns in record set '{example_rs}':")
    print(dataframes[example_rs].columns.tolist())
    dataframes[example_rs].head()
else:
    print('No record sets present.')

## 4. Exploratory Data Analysis (EDA)
Apply exploratory steps such as filtering, normalizing numeric fields, and grouping records by key attributes.

We'll choose a numeric field and a grouping field by their `@id` (as shown in the Data Overview). Please adapt these as appropriate for your data.

In [ ]:
# Choose a record set and select a numeric and group field (replace these IDs if your dataset has others)
record_set_id = record_set_ids[0] if len(record_set_ids) > 0 else None
df = dataframes[record_set_id]

# Detect likely numeric and group fields from the columns
print("Preview columns for numeric/group field selection:")
print(df.columns.tolist())

# Example field IDs (TO MODIFY: use the actual @id values for your dataset from earlier cell if different):
numeric_field_id = None
group_field_id = None
for col in df.columns:
    if numeric_field_id is None and ("age" in col.lower() or df[col].dtype in ['float64', 'int64']):
        numeric_field_id = col
    if group_field_id is None and ("sex" in col.lower() or "gender" in col.lower() or "msi" in col.lower()):
        group_field_id = col

print(f"Selected numeric field: {numeric_field_id}")
print(f"Selected group field: {group_field_id}")

if numeric_field_id:
    # Remove missing/non-numeric values
    numeric_series = pd.to_numeric(df[numeric_field_id], errors='coerce')
    threshold = numeric_series.mean() if not pd.isnull(numeric_series.mean()) else 0
    filtered_df = df[numeric_series > threshold].copy()
    print(f"\nFiltered records with {numeric_field_id} > average ({threshold:.2f}): {len(filtered_df)} records")
    # Normalize
    filtered_df[f"{numeric_field_id}_normalized"] = (numeric_series - numeric_series.mean()) / numeric_series.std()
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
else:
    print("No numeric field detected for EDA.")

# Group by group_field, if present
if group_field_id and group_field_id in filtered_df.columns:
    grouped = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
    print(f"\nGrouped by {group_field_id}, mean of {numeric_field_id}:")
    print(grouped)
else:
    print("No grouping field identified or present for EDA.")

## 5. Visualization
Visualize distributions or field relationships relevant for the dataset, such as histograms or boxplots.

**Note:** We use field `@id`, as always.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot numeric field distribution and compare by group if fields identified
if numeric_field_id:
    plt.figure(figsize=(8, 4))
    sns.histplot(filtered_df[numeric_field_id].dropna(), kde=True, bins=10, color='tab:blue')
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    if group_field_id and group_field_id in filtered_df.columns:
        plt.figure(figsize=(10, 5))
        sns.boxplot(data=filtered_df, x=group_field_id, y=numeric_field_id)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.show()
else:
    print("No numeric field detected for visualization.")

## 6. Conclusion
- We have loaded the [FAIR^2](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json) dataset using the Croissant schema and `mlcroissant`.
- All code operations (record sets, fields, columns) referenced entities exclusively by their unique Croissant `@id`.
- Tabular record sets were loaded, previewed, and basic exploratory steps such as filtering, normalization, grouping, and visualization were demonstrated—**all using only `@id` field references.**
- This workflow supports reproducible FAIR data science, leveraging schema-driven dataset documentation and robust programmatic access.

Feel free to modify field `@id` parameters as appropriate to focus your analyses.